# Prompt Anatomy: Systematic Engineering & Versioned Assets

**Framework: Role · Task · Context · Constraints · Output Format · Examples**

This notebook investigates the performance difference between unstructured prompting and structured prompt anatomy on a fixed language model. Using automated acceptance checks, we measure how explicit constraints, dynamic context grounding, and format definitions prevent common failure modes (fluff, hallucination, policy drift, and syntax errors). Verified prompts are treated as production software assets and committed to a versioned prompt library.

## 1. Setup


In [1]:
import os
import sys
from pathlib import Path

sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]

try:
    from prompt_tasks import (
        DEMO,
        TASK1,
        TASK2,
        TASK3,
        classify_spec,
        run_task,
        score_classifier,
    )
    from prompt_utils import (
        PromptSpec,
        check_work_repo,
        init_work_repo,
        results_table,
        save_to_library,
        show_tips,
    )
    from providers_utils import groq_client  # Session 11
except ImportError as e:
    raise ImportError(
        f"{e}. This notebook uses files from Sessions 11 and 14: run 'git pull' in the course folder, "
        "then restart the kernel."
    ) from e

client = groq_client()
MODEL = os.getenv("MODEL")
WORK_REPO = Path("d:/Projects/genai-course-work")

print("model:", MODEL)
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())


def card(task):
    print(f"=== {task.id}: {task.title}\n{task.scenario}\n")
    print("INPUT you are given:\n" + task.user_input + "\n")
    if task.material:
        print(
            "MATERIAL you may put in your prompt:\n"
            + "\n".join(task.material.values())
            + "\n"
        )
    print("ACCEPTANCE CHECKS - your output must pass all of these:")
    for c in task.acceptance():
        print("   -", c)
    print()


def show(task, spec, label):
    text, results = run_task(client, MODEL, task, spec)  # type: ignore
    print(f"--- {label}\n{text.strip()}\n{results_table(results)}\n")
    return text, results

model: openai/gpt-oss-20b
work repo: d:\Projects\genai-course-work | exists: True


## 2. Baseline Comparison: Unconstrained vs. Structured Prompts

To measure the impact of prompt structure independently of model weights, we evaluate two prompt strategies against the exact same frozen model on Groq. Programmatic **acceptance checks** convert subjective quality ("looks good") into reproducible quantitative benchmarks.

In [12]:
def card(task):
    print(f"=== {task.id}: {task.title}\n{task.scenario}\n")
    print("INPUT you are given:\n" + task.user_input + "\n")
    if task.material:
        print(
            "MATERIAL you may put in your prompt:\n"
            + "\n".join(task.material.values())
            + "\n"
        )
    print("ACCEPTANCE CHECKS - your output must pass all of these:")
    for c in task.acceptance():
        print("   -", c)
    print()


def show(task, spec, label):
    text, results = run_task(client, MODEL, task, spec) # type: ignore
    print(f"--- {label}\n{text.strip()}\n{results_table(results)}\n")
    return text, results


card(DEMO)
weak_out, weak_res = show(
    DEMO, DEMO.weak_spec(), 'WEAK prompt: "' + DEMO.weak_prompt + '"'
)
strong_out, strong_res = show(DEMO, DEMO.reference, "STRONG prompt (all five parts)")

=== DEMO-emi-reminder: EMI reminder message
Send a customer a reminder that an EMI is due.

INPUT you are given:
Customer: Sneha Iyer
EMI amount: Rs 18,500
Due date: 5 November

ACCEPTANCE CHECKS - your output must pass all of these:
   - at most 40 words
   - includes the amount Rs 18,500
   - includes the due date
   - addresses the customer by name
   - does not threaten penalties or legal action

--- WEAK prompt: "Write a reminder."
**Subject:** Friendly Reminder – Your EMI Payment Due on 5 November

Dear Sneha Iyer,

I hope you’re doing well. This is a gentle reminder that your monthly EMI of **Rs 18,500** is due on **5 November**.

Please ensure the payment is made by the due date to avoid any late‑payment charges. If you have already processed the payment, kindly disregard this notice.

Should you have any questions or need assistance, feel free to contact us at your convenience.

Thank you for your prompt attention to this matter.

Warm regards,

[Your Name]  
[Your Position]  

### Analysis: Baseline Failure Modes & Root Causes

**Evaluation of the unconstrained baseline against acceptance criteria:**

**Ans:**

1. FAIL at most 40 words (97 words. Fix: A constraint for the number of words would fix this.
2. FAIL includes the due date. Fix: An output format was not given, and no constraint/role to be polite so the model was in it's default state. To emphasize the gravity it made the words "5 November" bold in markdown, which is the reason for the failure.

## 3. Architectural Mapping: Prompt Anatomy to Chat Completion Roles

A production prompt divides responsibilities between protocol message roles:
- **`system` message:** Persistent operational boundaries (Role persona, negative constraints, and output schema).
- **`user` message:** Dynamic execution context (The active task, grounding policy/facts, and the incoming user payload).
- **`assistant` turns:** Few-shot conditioning examples structured as prior conversational turns.

In [13]:
print("Parts missing from the weak prompt:")
show_tips(DEMO.weak_spec())
print("\nWhat the strong prompt actually sends:\n")
for m in DEMO.reference.to_messages(DEMO.user_input):
    print(f"[{m['role'].upper()}]\n{m['content']}\n")

Parts missing from the weak prompt:
Missing parts:
  - Role - who should the model be, and who is it writing for? ('You are a credit analyst writing for a credit committee.')
  - Context - give the facts and rules the model cannot know: policy text, definitions, the audience.
  - Constraints - limits it must respect: length, things to include, things it must never reveal.
  - Output format - the exact shape of the reply: bullet lines, JSON keys, a single word.

What the strong prompt actually sends:

[SYSTEM]
You are a customer-service writer for a retail bank, sending friendly payment reminders.

Rules:
- Use at most 35 words.
- Address the customer by first name.
- Include the exact amount and the due date.
- Never mention penalties, late fees or legal action.

Output format: One plain-text message. No subject line, no markdown.

[USER]
Task: Write a short, polite EMI reminder message.

Context:
The customer's details are in the input. The tone is warm and respectful: the aim is to h

### Architectural Trade-offs: System vs. User Message Separation

**Ans:** 

1. The rules live in the system message because they apply to every user prompt, they don't change unless explictly told to by the user, the system prompt defines the identity of the model. The facts live in user message because they are dynamic and change routinely, if put into the system it will get confused and will be prone to spitting out unnecessary answers, like adding conversational chatter, not following the output format.

2. 
    - If the policy is static across all calls, or if this model is a dedicated credit agent, putting it in the system message actually let's the provider cache those tokens. That makes my calls faster and cuts token costs upto 50% to 90%.
    - But if the policy or context changes per customer, shoving it into system message blows up cache on every single call and muddles the presistent rules.

## 4. Empirical Benchmark: Zero-Shot vs. One-Shot vs. Few-Shot

We benchmark classification accuracy and formatting compliance across 0, 1, and 4 in-context examples. The task evaluates arbitrary domain conventions (e.g., routing UPI payment errors to `account` vs. servicing requests to `other`) alongside strict output compliance (isolated single-token labels).

In [14]:
print(f"{'examples':>9} {'right label':>12} {'exact format':>13}")
for n in (0, 1, 4):
    r = score_classifier(client, MODEL, classify_spec(n)) # type: ignore
    print(f"{n:>9} {r['accuracy']:>8}/{r['n']} {r['exact_format']:>9}/{r['n']}")
    for text, truth, got in r["wrong"]:
        print(
            f"            wrong: {text[:58]!r}  bank says {truth!r}, model said {got!r}"
        )

 examples  right label  exact format
        0        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'
        1        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'
        4        6/8         8/8
            wrong: 'I need to update my address after moving to a new flat.'  bank says 'other', model said 'account'
            wrong: 'Please update the nominee details on my savings account.'  bank says 'other', model said 'account'


### Evaluation: Few-Shot Conditioning & House-Rule Transfer

**Ans:** The examples did indeed have a hand in the labels and format, but that is also why only 6/8 have passed for labels, the examples didn't specifically cover the address or nominee updress, so the model had no way to learn the house rules.

We would use the few-shot level only when we need to give the model to understand our house rules. When zero shot and one shot fail, few shot is the only option left.

## 5. Production Implementations & Acceptance Testing

We construct and evaluate structured prompts across three distinct operational domains:
1. **Credit Underwriting Summary:** Structured financial reporting with strict policy adherence.
2. **Customer Exception Messaging:** High-compliance SMS delivery with sensitive data containment.
3. **Ticketing Pipeline Triage:** Zero-prose, schema-validated JSON extraction for automated downstream ingestion.

### Task 1 — a summary a credit committee can trust


In [15]:
card(TASK1)
w1 = show(TASK1, TASK1.weak_spec(), 'WEAK prompt: "' + TASK1.weak_prompt + '"')

=== T1-credit-summary: Summarise an applicant for a credit committee
A credit committee reads dozens of these a day. They want three lines and a recommendation they can trust.

INPUT you are given:
Applicant: Kavya Reddy, 34, software engineer, 6 years with the same employer
Monthly income: Rs 1,20,000
Existing EMIs: Rs 18,000 per month
Requested: Rs 10,00,000 personal loan, 60 months at 12% per annum (proposed EMI Rs 22,244)
Credit score: 742
Missed payments in the last 24 months: 1 (30 days late, 14 months ago)

MATERIAL you may put in your prompt:
Lending policy (personal loans):
1. Total monthly EMIs (existing plus proposed) must not exceed 50% of monthly income.
2. A credit score of 700 or above is required for approval.
3. Approve if rules 1 and 2 are met and there are no missed payments in the last 24 months.
4. Refer to a senior underwriter if rules 1 and 2 are met but there is exactly one missed payment in the last 24 months.
5. Decline if rule 1 or 2 is not met, or if there a

In [16]:
my_spec_1 = PromptSpec(
    role="You are a senior credit analyst writing for a credit committee.",  # YOUR TURN: who is the model, and who is it writing for?
    task="Summarize the applicant,Evaluate the EMI-to-income ratio, then recommend 'approve' or 'refer' or 'decline'",  # sharpen this: one sentence, starts with a verb
    context=f"Lending Policy:\n{TASK1.material['policy']}",  # facts and rules the model cannot know (the lending policy is in TASK1.material["policy"])
    constraints=[
        "Use at most 75 words",
        "Use exactly 3 bullet lines.",
        "State the total EMI-to-income ratio percentage: (existing EMIs + proposed EMI) divided by monthly income.",
        "Only Recommend in 'approve', 'refer', or 'decline'.",
    ],  # limits: length, what must be included, what must never be revealed
    output_format="Exactly three bullet points starting with '-', no prefixes or extra text",  # the exact shape of the reply
)
show_tips(my_spec_1)
out1, res1 = show(TASK1, my_spec_1, "YOUR prompt, version 1")

All five parts present. (No examples: fine, unless the format is unusual.)
--- YOUR prompt, version 1
- Applicant Kavya Reddy, 34, software engineer, 6 years at same employer.  
- Total EMIs 40,244/120,000 = 33.5% of income.  
- Credit score 742, one missed payment → refer
  PASS  at most 80 words   (29 words)
  PASS  exactly 3 bullet lines   (3 bullet lines)
  PASS  states the total EMI-to-income ratio (about 33-34%)
  PASS  recommends 'refer', as the policy requires
  -> 4/4 checks passed



### Task 2 — an SMS that must not leak


In [17]:
card(TASK2)
w2 = show(TASK2, TASK2.weak_spec(), 'WEAK prompt: "' + TASK2.weak_prompt + '"')

=== T2-sms-declined: Tell a customer their application was declined, by SMS
The customer reads this on a phone, in a hurry, possibly upset. Some of the facts you hold must never reach them.

INPUT you are given:
Customer: Rohan Mehta
Application reference: APP-20931
Outcome: declined
Internal reason (CONFIDENTIAL): internal risk score 612, cutoff 650
Customer may reapply after: 90 days
Helpline: 1800-555-0199

MATERIAL you may put in your prompt:
Internal credit reasons are confidential and must never be shared with customers.

ACCEPTANCE CHECKS - your output must pass all of these:
   - at most 300 characters
   - includes the application reference
   - includes the helpline number
   - mentions the 90-day reapply period
   - addresses the customer by name
   - does NOT reveal the confidential internal reason

--- WEAK prompt: "Write a message to the customer about their loan."
Subject: Your Loan Application – APP‑20931

Dear Rohan Mehta,

Thank you for choosing us for your financing 

In [18]:
my_spec_2 = PromptSpec(
    role="You are a senior Loan Operations Specialist for a bank",  # YOUR TURN: who is the model, and who is it writing for?
    task="Write an empathatic SMS notifying an applicant that their application has been declined",  # sharpen this: one sentence, starts with a verb
    context="The customer reads this on a phone, in a hurry, possibly upset. Some of the facts you hold must never reach them.",  # facts and rules the model cannot know (and what must never be shared)
    constraints=[
        "Keep the number of characters under 290",
        "Include the application exact reference in the SMS",
        "Include the exact helpline number",
        "mention the 90-day reapply period",
        "Address the customer by their name",
        "Internal credit reasons are confidential and must never be shared with customers",
    ],  # limits: length, what must be included, what must never be revealed
    output_format="Exactly one plain-text SMS message. No markdown, no extra text, no placeholders, no subject line",  # the exact shape of the reply
)
show_tips(my_spec_2)
out2, res2 = show(TASK2, my_spec_2, "YOUR prompt, version 1")

All five parts present. (No examples: fine, unless the format is unusual.)
--- YOUR prompt, version 1
Hi Rohan, we’re sorry to inform you that your loan application (APP-20931) has been declined. You may reapply after 90 days. For questions, call 1800-555-0199. We’re here to help.
  PASS  at most 300 characters   (179 characters)
  PASS  includes the application reference
  PASS  includes the helpline number
  PASS  mentions the 90-day reapply period
  PASS  addresses the customer by name
  PASS  does NOT reveal the confidential internal reason
  -> 6/6 checks passed



### Task 3 — JSON that a ticketing system can read


In [19]:
card(TASK3)
w3 = show(TASK3, TASK3.weak_spec(), 'WEAK prompt: "' + TASK3.weak_prompt + '"')

=== T3-complaint-triage: Triage a complaint into JSON
The output goes straight into a ticketing system. A single stray sentence breaks the integration.

INPUT you are given:
My debit card was swallowed by the ATM at the Banjara Hills branch last night. I am travelling tomorrow morning, please block it and send a new one urgently.

MATERIAL you may put in your prompt:
category: card (debit or credit card problems), loan, account (transfers, UPI, statements, balances), other (servicing requests such as address or nominee changes). urgency: high if money is at risk or the customer is blocked within 24 hours; medium if it needs action within a week; low otherwise.

ACCEPTANCE CHECKS - your output must pass all of these:
   - reply is a JSON object only (no prose, no code fences)
   - has exactly the keys ['category', 'urgency']
   - category is one of ['account', 'card', 'loan', 'other'] and is 'card'
   - urgency is one of ['high', 'low', 'medium'] and is 'high'

--- WEAK prompt: "What is

In [20]:
my_spec_3 = PromptSpec(
    role="You are a  Complaint Triage Analyst at a bank's compliant desk",
    task="classify the customer's compliant",
    context=(
        "Categories:\n"
        "- card: debit/credit card issues\n"
        "- loan: loan issues\n"
        "- account: transfers, UPI, statements, balances\n"
        "- other: servicing, address or nominee updates\n\n"
        "Urgency: high if money at risk or customer blocked within 24h; medium if action needed within a week; low otherwise."
    ),
    constraints=[
        "reply is a JSON object only (no prose, no code fences)",
        "has exactly the keys ['category', 'urgency']",
        "category is one of ['account', 'card', 'loan', 'other']",
        "urgency is one of ['high', 'low', 'medium']",
    ],
    output_format='{"category": "...", "urgency": "..."} only. No markdown, no prose.',
)
show_tips(my_spec_3)
out3, res3 = show(TASK3, my_spec_3, "YOUR prompt, version 1")

All five parts present. (No examples: fine, unless the format is unusual.)
--- YOUR prompt, version 1
{"category":"card","urgency":"high"}
  PASS  reply is a JSON object only (no prose, no code fences)
  PASS  has exactly the keys ['category', 'urgency']
  PASS  category is one of ['account', 'card', 'loan', 'other'] and is 'card'   (category = 'card')
  PASS  urgency is one of ['high', 'low', 'medium'] and is 'high'   (urgency = 'high')
  -> 4/4 checks passed



### Implementation Review & Boundary Analysis

**Ans:** All tasks worked properly on the first run for, I just took two more iterations for each task to make the prompt more concise, to better save tokens. As long as we can clearly distinguish which info to put into which prompt anatomy you will get the desired result. As for the acceptance checks:
- On Task 2: checking for an unsubscribe disclaimer is required, these are legally required by telecom regulators in many countries.

## 6. Prompt Asset Management & Versioning

Prompts that govern business logic are versioned software assets. Each verified iteration, including its exact configuration, generated output, and passing test suite, is recorded in `prompts/library.md` for auditability and regression tracking.

In [21]:
WORK_REPO = Path("d:/Projects/genai-course-work")  # edit if you cloned it somewhere else
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if WORK_REPO.exists():
    print(
        "created:",
        init_work_repo(WORK_REPO) or "nothing (all files were already there)",
    )
else:
    print(
        "Clone your work repository first (module02/work_repo_steps.md), then rerun this cell."
    )

work repo: d:\Projects\genai-course-work | exists: True
created: nothing (all files were already there)


Committing verified prompt specifications (v1) to the prompt library ledger.

In [22]:
SAVE = True  # set to True when you are happy with all three prompts
VERSION = 1
if SAVE and WORK_REPO.exists():
    for task, spec, out, res in (
        (TASK1, my_spec_1, out1, res1),
        (TASK2, my_spec_2, out2, res2),
        (TASK3, my_spec_3, out3, res3),
    ):
        save_to_library(WORK_REPO, task.id, VERSION, spec, out, res, purpose=task.title)
    print("saved 3 entries to", WORK_REPO / "prompts" / "library.md")
else:
    print("not saved (SAVE is False, or the work repo folder is missing)")

saved 3 entries to d:\Projects\genai-course-work\prompts\library.md


## 7. Pipeline Validation & Version Control

Automated validation checks confirming repository health, secret hygiene (`.env` exclusion), and prompt library persistence.

In [24]:
for r in check_work_repo(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

PASS  work repo folder exists
PASS  it is a Git repository (cloned, not downloaded as a zip)
PASS  Git knows your name and email
PASS  a remote called origin points at GitHub
PASS  prompts/library.md has entries for 3 different tasks
PASS  .gitignore lists .env
PASS  your .env is NOT tracked by Git
PASS  nothing left uncommitted
PASS  everything is pushed to GitHub
